# Neuromorphic computing — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
print('setup ok')

## Leaky integrate-and-fire dynamics
A leaky integrate-and-fire neuron is a tiny event-driven recurrent system. These examples show immediate spikes, delayed accumulation, the leak-memory tradeoff, weighted presynaptic input, and the sparse-work advantage.

In [ ]:
# 1) A strong event spikes immediately and resets
def lif(xs,lam=0.8,theta=1.0):
    v=0.; vals=[]; spikes=[]
    for x in xs:
        pre=lam*v+x; s=pre>=theta; vals.append(pre); spikes.append(int(s)); v=0. if s else pre
    return np.array(vals), np.array(spikes)
vals,sp=lif([1.2,0,0,0,0])
plt.figure(figsize=(6,3)); plt.stem(range(1,6),sp); plt.ylim(-0.1,1.2); plt.title('strong first input -> spike then reset')
assert vals[0]==1.2 and sp.tolist()==[1,0,0,0,0]

In [ ]:
# 2) Weak repeated inputs accumulate until threshold
vals,sp=lif([0.3,0.3,0.3,0.3,0.3])
plt.figure(figsize=(6,3)); plt.plot(range(1,6),vals,'-o'); plt.axhline(1,ls='--',c='r'); plt.title('weak evidence accumulates to a delayed spike')
plt.xlabel('time'); plt.ylabel('pre-reset voltage')
assert np.allclose(vals,[0.3,0.54,0.732,0.8856,1.00848]) and sp.tolist()==[0,0,0,0,1]

In [ ]:
# 3) The leak controls the memory window
xs=[0.3,0.3,0.3,0.3]; low,_=lif(xs,lam=0.2); high,_=lif(xs,lam=0.8)
plt.figure(figsize=(6,3)); plt.plot(low,'-o',label='lambda=0.2'); plt.plot(high,'-s',label='lambda=0.8'); plt.axhline(1,ls='--',c='gray')
plt.legend(); plt.title('higher retention preserves weak evidence')
assert np.allclose(low,[0.3,0.36,0.372,0.3744]) and np.allclose(high,[0.3,0.54,0.732,0.8856])

In [ ]:
# 4) Weighted presynaptic spikes form the current
x=np.array([1,0,1]); w=np.array([0.6,0.2,0.5]); current=float(x@w); theta=1.0
plt.figure(figsize=(6,3)); plt.bar(['x1*w1','x2*w2','x3*w3'],x*w); plt.axhline(theta,ls='--',c='r'); plt.title(f'weighted input sum = {current:.3f}')
assert abs(current-1.1)<1e-12 and current>=theta

In [ ]:
# 5) Sparse spikes skip dense work
n=1000; active=12; frac=active/n; skipped=1-frac
plt.figure(figsize=(6,3)); plt.bar(['active work','skipped work'],[frac,skipped],color=['tab:orange','tab:green']); plt.ylim(0,1); plt.title('event-driven layer does work only on spikes')
assert abs(frac-0.012)<1e-12 and abs(skipped-0.988)<1e-12